# Ontologie AIF.owl -- l'architecture Argumentum des sophismes (OWL2 + annotation-space)

> **EPIC #4960 -- Volet A, etape 1** (socle ontologique Argument_Analysis). Issue source : [#5721](https://github.com/jsboige/CoursIA/issues/5721).

> **⚠️ Re-fondation 2026-07-10 (Argumentum PR #763)** — le generateur OWL des Fallacies a change d'idiome. Les concepts ne sont plus des `NamedIndividual` mais des **`owl:Class`**, et **toutes** les relations (hierarchie SKOS, crossLink, AIF attack) sont serialisees en **`AnnotationAssertion`** (`SKOSHelper` d'OWLSharp etant casse sur cette version). Ce notebook a ete re-fonde en consequence : le parseur regex tolerant reste la bonne approche (rdflib et owlready2 echouent toujours), mais **les structures extraites changent**.

Ce notebook explore `ontologies/argumentum_fallacies.owl` (OWL2/XML, ~6,0 MB) et en extrait la structure sans passer par un parseur RDF strict.

**Objectifs d'apprentissage** :

1. Charger l'OWL2/XML via un **parseur regex tolerant** et comprendre *pourquoi* rdflib/owlready2 echouent sur ce dialecte.
2. Compter la structure du **nouvel idiome annotation-space** : **1509 concepts `owl:Class`** (+ 4 classes AIF RA/I/CA-node + Conflict), **10 `ObjectProperty`**, **18 284 `AnnotationAssertion`** (7 773 resource + 10 511 literal).
3. Extraire les **labels multilingues** (`skos:prefLabel`, 2 816 = 1 408 FR + 1 408 EN) et reperer les **schemes de Walton** (100 classes de forme `*_Inference_*` materialisees depuis les mappings AIF_skos).
4. Inventorier les **relations** (AnnotationAssertion-resource) : hierarchie (`broader`/`narrower`/`inScheme`), **crossLink** (`mirrors`/`isRelatedTo`/`leverages`/… = 1 977) et **AIF attack** (`aifAttackedNode` = 93 → RA/I/CA-node).
5. Construire le **sous-graphe de l'Equivoque** (`semanticAmbiguity`) et montrer qu'il est desormais **type par AIF** (`aifAttackedNode → RA-node`).
6. Trois exercices formels (regle #2161).

**Sommaire** :

1. Contexte : Argumentum, OWL2 et l'idiome annotation-space (#763)
2. Charger l'ontologie : regex tolerant (rdflib echoue)
3. Structure : Class + ObjectProperty + AnnotationAssertion
4. Schemes de Walton (100 classes materialisees + labels)
5. Relations : inventaire des AnnotationAssertion-resource (crossLink + AIF)
6. Grappe Equivoque : sous-graphe type AIF
7. Exercices
8. Ponts avec la serie

## 1. Contexte : Argumentum, OWL2 et l'idiome annotation-space (#763)

L'ontologie **Argumentum** (https://www.argumentum.games) encode une taxonomie des sophismes (fallacies) et des vertus argumentatives a destination du jeu serieux *Argumentum*. Le fichier `argumentum_fallacies.owl` est genere par le pipeline .NET (`OwlGeneratorConfig.cs`) a partir du CSV canonique.

**Idiome du generateur (apres PR #763, mergee 2026-07-10)** :

- Chaque sophisme est une **`owl:Class`** (`<Declaration><Class IRI="…#semanticAmbiguity"/></Declaration>`) — et non plus un `NamedIndividual` comme dans l'ancien generateur. Le modele AIF (`RA-node`/`I-node`/`CA-node`/`Conflict`) apporte 4 classes supplementaires (namespace `arg.dundee.ac.uk/aif`).
- **Toutes** les relations passent par **`AnnotationAssertion`** : `SKOSHelper` d'OWLSharp est casse sur cette version, donc le generateur emet la hierarchie SKOS (`broader`/`narrower`/`inScheme`), les 8 verbes **crossLink** et les 2 colonnes **AIF attack** comme des annotations (resource pour un lien concept→concept, literal pour une valeur textuelle).
- Les IRI sont serialises en **forme complete** (`IRI="…"`), plus en `abbreviatedIRI="skos:prefLabel"` — d'ou la reecriture des regex de ce notebook.

**Pourquoi un parseur regex ?** rdflib echoue au parse (dialecte OWL2/XML + annotations resource non standard) et owlready2 charge le fichier mais **n'expose 0 classe** via son API Python. Le parseur regex tolerant reste le chemin fiable pour extraire la structure — c'est l'objet pedagogique de ce notebook.

> Le notebook frere `Argument_Analysis_Ontology_CrossLinks.ipynb` (PR-B) lit le **CSV canonique** ; les deux substances (OWL et CSV) ont ete **reconciliees par #763** (crossLink + AIF desormais emis dans l'OWL).

## 2. Charger l'ontologie : regex tolerant (rdflib echoue)

In [1]:
# --- Imports + parseur regex maison de l'OWL2/XML ---
from pathlib import Path
import re

OWL_PATH = Path("ontologies/argumentum_fallacies.owl").resolve()
print(f"Chargement de : {OWL_PATH}")
print(f"Existe : {OWL_PATH.exists()} (taille = {OWL_PATH.stat().st_size:,} octets)")

owl_text = OWL_PATH.read_text(encoding="utf-8")
print(f"Longueur du texte : {len(owl_text):,} caracteres, {owl_text.count(chr(10)) + 1:,} lignes")

# Axiomes de cardinalite : desormais bien formes (avec onProperty) -- ils ne sont
# plus le point de rupture (l'ancien OWL avait 37 axiom sans onProperty ligne ~4090).
n_obj_exact = len(re.findall(r"<ObjectExactCardinality", owl_text))
n_data_exact = len(re.findall(r"<DataExactCardinality", owl_text))
print(f"\nCardinalite : ObjectExactCardinality={n_obj_exact}, DataExactCardinality={n_data_exact} (bien formes).")

# Test rdflib : echoue toujours sur ce dialecte, mais pour une autre cause qu'avant #763.
try:
    import rdflib
    try:
        g = rdflib.Graph(); g.parse(str(OWL_PATH))
        print(f"rdflib : OK inattendu ({len(g):,} triples)")
    except Exception as e:
        print(f"rdflib : installe mais ECHEC au parse -> {type(e).__name__}: {str(e)[:80]}")
except ImportError:
    print("rdflib : non installe dans ce kernel (test skippe) -- historiquement il echoue sur ce dialecte.")

print("owlready2 : charge le fichier mais expose 0 classe via l'API Python (idiome annotation-space).")
print("\n=> On conserve le parseur regex tolerant : chemin fiable pour extraire la structure.")

Chargement de : D:\CoursIA\.claude\worktrees\argumentum-ontology-refresh\MyIA.AI.Notebooks\SymbolicAI\Argument_Analysis\ontologies\argumentum_fallacies.owl
Existe : True (taille = 6,000,847 octets)
Longueur du texte : 5,892,331 caracteres, 96,770 lignes

Cardinalite : ObjectExactCardinality=36, DataExactCardinality=0 (bien formes).


rdflib : installe mais ECHEC au parse -> TypeError: sequence item 0: expected str instance, NoneType found
owlready2 : charge le fichier mais expose 0 classe via l'API Python (idiome annotation-space).

=> On conserve le parseur regex tolerant : chemin fiable pour extraire la structure.


**Lecture** : le fichier OWL pese ~6,0 MB (~96 800 lignes). Les axiomes `ObjectExactCardinality` (36) sont desormais **bien formes** (avec `onProperty`) — ce ne sont plus les « 37 axiom casses ligne 4090 » de l'ancien snapshot. Malgre cela, **rdflib echoue toujours** (autre cause : le dialecte OWL2/XML avec annotations-resource) et **owlready2 charge mais n'expose rien** via son API Python. Le parseur regex tolerant reste donc l'approche retenue : il extrait Class, ObjectProperty et AnnotationAssertion directement du XML, sans dependre d'un moteur RDF.

## 3. Structure : Class + ObjectProperty + AnnotationAssertion

Depuis #763, l'ontologie Argumentum suit un idiome **annotation-space** :

| Couche | Pattern OWL2 | Compte | Role |
|--------|--------------|--------|------|
| Concepts | `Declaration/Class` | **1 509** (+ 4 AIF) | Un `owl:Class` par sophisme + RA/I/CA-node + Conflict |
| Predicats | `Declaration/ObjectProperty` | **10** | 8 verbes crossLink + `hasConflictedElement` + `aifAttackedNode` |
| Relations | `AnnotationAssertion` (resource) | **7 773** | hierarchie SKOS + crossLink + AIF attack (concept→concept) |
| Annotations | `AnnotationAssertion` (literal) | **10 511** | labels, definitions, exemples, `aifAttackType`… |

**Rupture avec l'ancien generateur** : `NamedIndividual`, `ClassAssertion` et `ObjectPropertyAssertion` ont **disparu** (0 occurrence). Là où l'ancien OWL materialisait chaque label multilingue comme un individu distinct (~10 976 `NamedIndividual`), le nouveau modele **1 `owl:Class` par sophisme** portant ses labels en annotations — d'ou la chute apparente du nombre de concepts (10 976 → 1 509), qui est en réalité une **normalisation** (plus de doublons par langue).

**Nouveaute #763** : l'AIF n'est plus seulement *declaree*, elle **type** les sophismes — 93 concepts portent `aifAttackedNode → RA/I/CA-node` (§5-6).

In [2]:
# --- Comptage de la structure : Class + ObjectProperty + AnnotationAssertion ---
from collections import Counter

# Concepts = Declaration/Class (le generateur #763 modelise 1 owl:Class par sophisme).
class_decl_pattern = re.compile(r'<Declaration>\s*<Class IRI="([^"]+)"\s*/>\s*</Declaration>')
class_decls = class_decl_pattern.findall(owl_text)
aif_classes = [c for c in class_decls if 'arg.dundee.ac.uk/aif' in c]
arg_classes = [c for c in class_decls if 'argumentum_fallacies' in c]
print(f"Declarations owl:Class : {len(class_decls):,}")
print(f"  - concepts Argumentum : {len(arg_classes):,}")
print(f"  - classes AIF : {len(aif_classes)}  {[c.split('#')[-1] for c in aif_classes]}")

# Predicats = Declaration/ObjectProperty (verbes crossLink + AIF).
op_pattern = re.compile(r'<Declaration>\s*<ObjectProperty IRI="([^"]+)"\s*/>\s*</Declaration>')
obj_props = op_pattern.findall(owl_text)
print(f"\nObjectProperty declarees : {len(obj_props)}")
print(f"  {[o.split('#')[-1] for o in obj_props]}")

# Ancien idiome (individus) -- desormais ABSENT.
n_named = len(re.findall(r'<NamedIndividual\s', owl_text))
n_class_assertion = len(re.findall(r'<ClassAssertion', owl_text))
n_opa = len(re.findall(r'<ObjectPropertyAssertion', owl_text))
print(f"\nAncien idiome (individus), desormais absent :")
print(f"  NamedIndividual         : {n_named}")
print(f"  ClassAssertion          : {n_class_assertion}")
print(f"  ObjectPropertyAssertion : {n_opa}")

# Relations = AnnotationAssertion (resource = concept->concept, literal = valeur textuelle).
aa_resource = re.compile(
    r'<AnnotationAssertion>\s*<AnnotationProperty IRI="([^"]+)"\s*/>\s*<IRI>([^<]+)</IRI>\s*<IRI>([^<]+)</IRI>\s*</AnnotationAssertion>')
aa_literal = re.compile(
    r'<AnnotationAssertion>\s*<AnnotationProperty IRI="([^"]+)"\s*/>\s*<IRI>([^<]+)</IRI>\s*<Literal(?:\s+[^>]*)?>([^<]*)</Literal>\s*</AnnotationAssertion>')
AA_RES = aa_resource.findall(owl_text)   # (predicate_iri, subject_iri, object_iri)
AA_LIT = aa_literal.findall(owl_text)    # (predicate_iri, subject_iri, literal_value)
print(f"\nAnnotationAssertion : {len(AA_RES) + len(AA_LIT):,}  (resource={len(AA_RES):,}, literal={len(AA_LIT):,})")

# Compteurs conserves pour les sections suivantes.
N_CLASSES = len(class_decls)
N_ARG_CLASSES = len(arg_classes)
N_OBJ_PROP = len(obj_props)

Declarations owl:Class : 1,513
  - concepts Argumentum : 1,509
  - classes AIF : 4  ['Conflict', 'I-node', 'RA-node', 'CA-node']

ObjectProperty declarees : 10
  ['hasConflictedElement', 'predatesOn', 'denounces', 'leverages', 'allows', 'opposes', 'inverts', 'mirrors', 'isRelatedTo', 'aifAttackedNode']

Ancien idiome (individus), desormais absent :
  NamedIndividual         : 0
  ClassAssertion          : 0
  ObjectPropertyAssertion : 0

AnnotationAssertion : 18,284  (resource=7,773, literal=10,511)


**Lecture** : le coeur de l'ontologie est desormais fait de **1 509 `owl:Class`** (concepts Argumentum) + 4 classes AIF, relies par **7 773 `AnnotationAssertion`-resource** et decrits par **10 511 `AnnotationAssertion`-literal**. Les `NamedIndividual`/`ClassAssertion`/`ObjectPropertyAssertion` de l'ancien generateur sont a **0** : #763 a bascule d'un modele *individus + triplets ObjectProperty* vers un modele *classes + annotation-space*. Les 10 `ObjectProperty` declarees (8 verbes crossLink + `hasConflictedElement` + `aifAttackedNode`) sont *declarees* comme proprietes mais *emises* comme annotations (contrainte de l'adapter OWLSharp).

## 4. Schemes de Walton : classes materialisees + labels

Les **schemes d'argumentation** (Walton, Reed & Macagno 2008) sont des patterns d'inference reconnus en logique informelle. Le generateur #763 **materialise** les mappings `AIF_skos*` du CSV en **100 classes de forme `*_Inference_*` / `*_Conflict`** (ex. `PopularOpinion_Inference_Conflict`, `Example_Inference_Conflicted`, `CauseToEffect_Inference`). On les repere directement dans les declarations de Class, en complement de la detection classique par substring dans les `skos:prefLabel` (2 816 labels = 1 408 FR + 1 408 EN).

In [3]:
# --- Labels multilingues (skos:prefLabel) + schemes de Walton ---
from collections import Counter

# prefLabel : AnnotationProperty IRI complet (plus d'abbreviatedIRI depuis #763).
pref_pattern = re.compile(
    r'<AnnotationAssertion>\s*<AnnotationProperty IRI="http://www\.w3\.org/2004/02/skos/core#prefLabel"\s*/>\s*<IRI>([^<]+)</IRI>\s*<Literal(?:\s+xml:lang="([^"]+)")?[^>]*>([^<]*)</Literal>\s*</AnnotationAssertion>')
pref_matches = pref_pattern.findall(owl_text)
print(f"skos:prefLabel extraits : {len(pref_matches):,}")
print(f"Langues : {dict(Counter(l.upper() for _, l, _ in pref_matches if l))}")

# dict {localname: {iri, prefLabel:[...]}}
taxonomy = {}
for iri, lang, label in pref_matches:
    local = iri.rstrip('#').split('#')[-1].split('/')[-1]
    if not local:
        continue
    entry = taxonomy.setdefault(local, {'iri': iri, 'prefLabel': []})
    if label:
        entry['prefLabel'].append(label)
print(f"Concepts distincts (par localname) : {len(taxonomy):,}")

# Schemes Walton materialises en classes (#763) -- signal robuste, sans faux positifs de substring.
walton_shaped = sorted(c.split('#')[-1] for c in arg_classes if '_Inference' in c or '_Conflict' in c)
print(f"\nClasses de forme Walton-scheme (materialisation AIF_skos) : {len(walton_shaped)}")
for c in walton_shaped[:8]:
    print(f"  - {c}")

# Detection complementaire par substring dans les prefLabel.
WALTON_SCHEMES = ["Position to Know", "Sign", "Cause to Effect", "Analogy", "Expert Opinion", "Popular Opinion"]
walton_found = {k: [] for k in WALTON_SCHEMES}
for local, entry in taxonomy.items():
    for label in entry.get('prefLabel', []):
        for k in WALTON_SCHEMES:
            if k.lower() in label.lower():
                walton_found[k].append((local, label)); break
print("\nSchemes Walton detectes dans les prefLabel :")
for k in WALTON_SCHEMES:
    hits = walton_found[k]
    print(f"  {'v' if hits else 'x'} {k:18s} : {len(hits)} occurrence(s)")

skos:prefLabel extraits : 2,816
Langues : {'FR': 1408, 'EN': 1408}


Concepts distincts (par localname) : 1,306

Classes de forme Walton-scheme (materialisation AIF_skos) : 100
  - Analogy_Inference_Conflict
  - Analogy_Inference_Conflicted
  - CausalSlipperySlope_Inference_Conflict
  - CausalSlipperySlope_Inference_Conflicted
  - CauseToEffect_Inference_Conflict
  - CauseToEffect_Inference_Conflicted
  - CircumstantialAdHominem_Inference_Conflict
  - CircumstantialAdHominem_Inference_Conflicted

Schemes Walton detectes dans les prefLabel :
  x Position to Know   : 0 occurrence(s)
  v Sign               : 11 occurrence(s)
  x Cause to Effect    : 0 occurrence(s)
  v Analogy            : 4 occurrence(s)
  x Expert Opinion     : 0 occurrence(s)
  x Popular Opinion    : 0 occurrence(s)


**Lecture** : **2 816 `skos:prefLabel`** (1 408 FR + 1 408 EN, soit 1 label par sophisme et par langue — la normalisation attendue du nouvel idiome). La detection des schemes de Walton s'appuie desormais sur les **100 classes materialisees** (`*_Inference_*`), signal bien plus robuste que la recherche par substring dans les labels (qui produisait des faux positifs sur « Sign », « Rule »…). La detection par substring est conservee a titre de comparaison pedagogique.

## 5. Relations : inventaire des AnnotationAssertion-resource

Depuis #763, les relations concept→concept sont portees par des **`AnnotationAssertion`-resource** (et non plus des `ObjectPropertyAssertion`). L'inventaire par predicat revele **trois familles** :

- **hierarchie SKOS** : `broader` (1 407), `narrower` (1 407), `inScheme` (1 408), `type` (1 409) ;
- **crossLink** (transverse, #141/#763) : `mirrors` (720), `isRelatedTo` (642), `leverages` (402), `inverts` (82), `allows` (66), `opposes` (50), `predatesOn` (13), `denounces` (2) = **1 977** au total ;
- **AIF attack** : `aifAttackedNode` (93) → RA/I/CA-node ;
- **mapping Walton** : `broadMatch` (57), `closeMatch` (10), `narrowMatch` (3) = 70.

**Inversion de la note historique** : l'ancien OWL **n'incluait PAS** les 8 verbes crossLink (ils etaient CSV-only). #763 les a **cables et emis** — ce notebook les inventorie donc desormais dans l'OWL.

In [4]:
# --- Inventaire des relations (AnnotationAssertion-resource) par predicat ---
from collections import Counter

rel_usage = Counter()
for pred_iri, _subj, _obj in AA_RES:
    rel_usage[pred_iri.rstrip('#').split('#')[-1].split('/')[-1]] += 1

print("Relations (AnnotationAssertion-resource) par predicat :")
print(f"  {'Predicat':22s} | Count")
print("  " + "-" * 34)
for pred, n in rel_usage.most_common(20):
    print(f"  {pred:22s} | {n:>5d}")
print(f"\n  Total predicats distincts : {len(rel_usage)}")

# Les 8 verbes crossLink SONT desormais dans l'OWL (#763) -- inversion de la note historique.
CROSSLINK_VERBS = ['predatesOn', 'denounces', 'leverages', 'allows', 'opposes', 'inverts', 'mirrors', 'isRelatedTo']
n_crosslink = sum(rel_usage[v] for v in CROSSLINK_VERBS)
print(f"\n=> crossLink dans l'OWL : {n_crosslink:,} AnnotationAssertion")
print(f"   (mirrors={rel_usage['mirrors']}, isRelatedTo={rel_usage['isRelatedTo']}, "
      f"leverages={rel_usage['leverages']}, inverts={rel_usage['inverts']}, ...)")
print(f"   AIF attack : aifAttackedNode={rel_usage['aifAttackedNode']} (resource -> RA/I/CA-node)")
print(f"   mapping Walton : broadMatch={rel_usage['broadMatch']}, closeMatch={rel_usage['closeMatch']}, "
      f"narrowMatch={rel_usage['narrowMatch']}")
print("\n   NOTE : contrairement a l'ancien OWL (crossLink ABSENT), #763 a cable ces 8 verbes.")

Relations (AnnotationAssertion-resource) par predicat :
  Predicat               | Count
  ----------------------------------
  type                   |  1409
  inScheme               |  1408
  narrower               |  1407
  broader                |  1407
  mirrors                |   720
  isRelatedTo            |   642
  leverages              |   402
  aifAttackedNode        |    93
  inverts                |    82
  allows                 |    66
  broadMatch             |    57
  opposes                |    50
  predatesOn             |    13
  closeMatch             |    10
  narrowMatch            |     3
  denounces              |     2
  hasTopConcept          |     1
  topConceptOf           |     1

  Total predicats distincts : 18

=> crossLink dans l'OWL : 1,977 AnnotationAssertion
   (mirrors=720, isRelatedTo=642, leverages=402, inverts=82, ...)
   AIF attack : aifAttackedNode=93 (resource -> RA/I/CA-node)
   mapping Walton : broadMatch=57, closeMatch=10, narrowMatch=3



**Lecture** : l'inventaire par predicat montre la structure reelle de la taxonomie. La colonne vertebrale hierarchique (`broader`/`narrower`/`inScheme`/`type`, ~1 408 chacune = une par sophisme) est desormais **doublee d'une couche transverse crossLink** (1 977 relations, dominee par `mirrors` et `isRelatedTo`) et d'une **couche AIF attack** (93 `aifAttackedNode`). C'est le changement de fond apporte par #763 : la taxonomie n'est plus un simple arbre, mais un **graphe** hierarchie + transverse + AIF.

## 6. Grappe Equivoque : sous-graphe type AIF

L'**Equivoque** (Equivocation) est un sophisme d'ambiguite : un meme terme employe dans des sens differents au cours d'un meme raisonnement. Dans l'ontologie, le concept `semanticAmbiguity` porte **11 relations** (AnnotationAssertion-resource) :

- hierarchie : `broader → ambiguity`, `narrower → {vagueness, polysemicLexicalShift, polysemyBySemanticChange}`, `inScheme → fallacyScheme`, `type → Concept` ;
- **AIF attack** : `aifAttackedNode → RA-node` — le sophisme est formellement modelise comme une **attaque de l'inference** (RA-node = Rule-Application node au sens AIF, ce qui correspond a un *undercut* ASPIC+).

C'est la demonstration concrete de #763 : **l'AIF est desormais utilisee pour typer les sophismes**, la ou l'ancien OWL se contentait de *declarer* les nodes AIF sans les relier.

In [5]:
# --- Sous-graphe autour de semanticAmbiguity via AnnotationAssertion-resource ---

# Noeuds lies a l'Equivoque (label contenant equivoc/quivoque).
equiv_nodes = []
for local, entry in taxonomy.items():
    for label in entry.get('prefLabel', []):
        if 'quivoque' in label.lower() or 'equivoc' in label.lower():
            equiv_nodes.append((local, label, entry.get('iri'))); break
print(f"Noeuds lies a l'Equivoque : {len(equiv_nodes)}")
for local, label, iri in equiv_nodes:
    print(f"  - {local:30s} -> {label}")

# Sous-graphe AnnotationAssertion-resource autour de semanticAmbiguity.
TARGET = "semanticAmbiguity"
print(f"\nSous-graphe (AnnotationAssertion-resource) autour de '{TARGET}' :")
G_equiv = {}
for pred_iri, subj_iri, obj_iri in AA_RES:
    subj = subj_iri.rstrip('#').split('#')[-1].split('/')[-1]
    obj = obj_iri.rstrip('#').split('#')[-1].split('/')[-1]
    pred = pred_iri.rstrip('#').split('#')[-1].split('/')[-1]
    if subj == TARGET or obj == TARGET:
        G_equiv[(subj, pred, obj)] = (pred_iri, subj_iri, obj_iri)

print(f"  Relations impliquees : {len(G_equiv)}")
nodes = set([k[0] for k in G_equiv] + [k[2] for k in G_equiv])
print(f"  Noeuds impliques : {len(nodes)}")
print("\nTriplets :")
for (subj, pred, obj), _ in G_equiv.items():
    tag = "  <-- AIF attack" if pred == 'aifAttackedNode' else ""
    print(f"  {subj[:32]:32s} --[{pred:16s}]--> {obj[:24]:24s}{tag}")

Noeuds lies a l'Equivoque : 3
  - antitheticalEquivocation       -> Équivoque antithétique
  - semanticAmbiguity              -> Équivoque
  - takingAdvantageOfTheNaysayer   -> Équivoque antithétique

Sous-graphe (AnnotationAssertion-resource) autour de 'semanticAmbiguity' :
  Relations impliquees : 11
  Noeuds impliques : 8

Triplets :
  semanticAmbiguity                --[type            ]--> Concept                 
  semanticAmbiguity                --[inScheme        ]--> fallacyScheme           
  ambiguity                        --[narrower        ]--> semanticAmbiguity       
  semanticAmbiguity                --[broader         ]--> ambiguity               
  semanticAmbiguity                --[narrower        ]--> vagueness               
  vagueness                        --[broader         ]--> semanticAmbiguity       
  semanticAmbiguity                --[narrower        ]--> polysemicLexicalShift   
  polysemicLexicalShift            --[broader         ]--> semanticAmbigu

**Lecture** : le sous-graphe de `semanticAmbiguity` combine sa **hierarchie** (parent `ambiguity`, enfants `vagueness` / `polysemicLexicalShift` / `polysemyBySemanticChange`) et son **typage AIF** (`aifAttackedNode → RA-node`). Le degre du noeud (11) melange donc des aretes taxonomiques et une arete AIF — c'est exactement la reconciliation que #763 apporte : une même entite porte simultanement sa place dans l'arbre SKOS et son role dans le modele d'attaque AIF.

## 7. Exercices

Trois exercices formels (regle #2161 : >= 3 exos par notebook pedagogique). Chaque exercice est un *stub* `print("Exercice a completer")` que l'etudiant complete.

### Exercice 1 -- Distribution des concepts par prefixe de scheme

A partir de `arg_classes` (liste d'IRI de `owl:Class` Argumentum) et de `taxonomy` (dict localname -> labels), calculer combien de concepts sont des **classes materialisees de scheme** (`*_Inference_*` / `*_Conflict`) versus des sophismes simples. Renvoyer un `dict` {categorie: compte} et l'afficher trie.

In [6]:
# Exercice 1 a completer
# TODO etudiant : a partir de `arg_classes` (IRI de owl:Class) et de `taxonomy`
# (dict localname -> labels), separer les classes materialisees de scheme
# (`_Inference` / `_Conflict`) des sophismes simples. Renvoyer un dict {cat: n}.
print("Exercice a completer")

Exercice a completer


### Exercice 2 -- Visualisation matplotlib du sous-graphe Equivoque

Reprendre `G_equiv` de la section 6 et le visualiser avec `matplotlib` + `networkx.spring_layout` (seed=42). Colorer les noeuds par degre (entrant + sortant), afficher les labels des proprietes sur les aretes.

*Indice* : construire un `nx.DiGraph`, ajouter les aretes depuis `G_equiv.items()`. Si `G_equiv` est vide, verifier que `semanticAmbiguity` est bien le `TARGET` (il peut etre absent si la taxonomie change).

In [7]:
# Exercice 2 a completer
# TODO etudiant : reprendre G_equiv de la section 6, construire un nx.DiGraph,
# spring_layout (seed=42), couleur des noeuds par degre, edge_labels affiches.
# Si matplotlib n'est pas disponible, fallback ASCII (cf. §6).
print("Exercice a completer")

Exercice a completer


### Exercice 3 -- Detection des sophismes "Ambiguite" et hierarchie

Implementer une fonction `trouver_ambiguite(taxonomy)` qui retourne tous les concepts dont le `prefLabel` fr/en mentionne "ambig", "equivoc", "quivoqu". Renvoyer un `dict {local: [labels_matches]}`. Faux positifs a eviter : "ambiguite du contexte" hors sophisme. Bonus : croiser avec `AA_RES` pour afficher, pour chaque match, son parent `broader`.

In [8]:
# Exercice 3 a completer
# TODO etudiant : implementer trouver_ambiguite(taxonomy) qui retourne
# un dict {local: [labels_matches]} sur les prefLabel fr/en. Bonus : croiser
# avec AA_RES (predicat 'broader') pour afficher le parent de chaque match.
print("Exercice a completer")

Exercice a completer


## 8. Ponts avec la serie

| Direction | Lien | Relation |
|-----------|------|----------|
| <-> ArgumentProfile | `Argument_Analysis_ArgumentProfile.ipynb` | Vue d'ensemble de la serie, introduction aux schemes |
| <-> CrossLinks (PR-B) | `Argument_Analysis_Ontology_CrossLinks.ipynb` | Substance CSV canonique — **reconciliee avec cet OWL par #763** |
| -> SemanticKernel | `SemanticKernel/*` | Exploitation des schemes pour l'analyse d'arguments |

### Honnetete methodologique

1. **Parseur regex, pas un moteur RDF** : rdflib echoue au parse et owlready2 expose 0 classe via l'API Python. Le parseur regex est robuste pour *extraire* la structure mais ne fait **pas** de raisonnement (pas d'inference de subsomption, pas de fermeture transitive). Pour un raisonnement OWL complet, il faudrait corriger l'idiome de serialisation en amont (generateur Argumentum).

2. **L'OWL utilise desormais l'AIF** (inversion de la version precedente de ce notebook) : 93 concepts portent `aifAttackedNode → RA/I/CA-node` (undercut→RA, undermine→I, rebut→CA, ratifie #707§4). La note historique « l'OWL n'utilise PAS l'AIF » est **obsolete**.

3. **Les crossLink sont dans l'OWL** (inversion) : 1 977 `AnnotationAssertion` (mirrors/isRelatedTo/leverages/…). La note historique « crossLink_* CSV-only, hors-repo » est **obsolete** — #763 a cable le chemin CSV→OWL.

4. **Ontologie soeur presente** : `argumentum_virtues.owl` est desormais dans `ontologies/` (aux cotes de `argumentum_fallacies.owl`). La note historique « argumentum_virtues.owl absente » est **obsolete**.

5. **Idiome annotation-space** : toutes les relations sont des `AnnotationAssertion` (et non des `ObjectPropertyAssertion`), consequence du bug `SKOSHelper` d'OWLSharp. C'est un choix de serialisation du generateur, pas une limite de modelisation — les 10 `ObjectProperty` sont bien declarees.